# C10 prereg55：c49 配方換種子 1，320,000 步（Colab G4）

執行階段選 G4（RTX PRO 6000；執行階段版本 2026.07），按「執行階段 → 全部執行」。不用改任何程式。
套件和 prereg51 B 臂共用（C10/p42 kit、C10/p51/overlay_parts）；輸出在 C10/p55/S/，每 5,000 步存檔，斷線後重按「全部執行」會接續。
練完打包 C10/p55/C10_p55_S_results.zip。任何一格出現紅字或 STOP，停下來，截最後 20 行。

In [ ]:
# 0. 把 Drive 上 C10/p42/kit_parts/ 的 66 份接回 C10_p42_kit.zip（已存在且雜湊對就略過）
import hashlib, os, glob
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/C10/p42'
KIT = os.path.join(D, 'C10_p42_kit.zip')
SHA = '6f5c93650375db2d13a3182d5c2808902b679686975438486a0342e5ad5a7306'
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''):
            h.update(b)
    return h.hexdigest()
if not (os.path.exists(KIT) and sha(KIT) == SHA):
    parts = sorted(glob.glob(os.path.join(D, 'kit_parts', 'kit.part*')))
    assert len(parts) == 66, 'STOP: kit_parts 只有 %d 份' % len(parts)
    tmp = '/content/C10_p42_kit.zip'
    with open(tmp, 'wb') as o:
        for p in parts:
            with open(p, 'rb') as f:
                o.write(f.read())
    assert sha(tmp) == SHA, 'STOP: 接回的 zip 雜湊不對'
    import shutil; shutil.copyfile(tmp, KIT)
    assert sha(KIT) == SHA, 'STOP: 複製到 Drive 後雜湊不對'
print('kit OK', SHA[:12])


In [ ]:
# 1. 雲端硬碟、GPU、套件（不用改）
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')
DRIVE, WORK, VARIANT = '/content/drive/MyDrive/C10/p42', '/content/c10', 'V2'
KIT_SHA, FILES = '6f5c93650375db2d13a3182d5c2808902b679686975438486a0342e5ad5a7306', {"data/prereg06/feats/meta.npz": "b1b5def7312dc6321ab08dd7e74e9b9069c49e9efb8caf7133b9869bf7338df5", "data/prereg36/cache/cache.json": "764a61937e60ab660972ecbe8a585e4205bd27a132ce74a64f4cc7390533c48a", "data/prereg36/cache/codes.npy": "be5831abbccf22be3c96725082939638e6607f88ad45b1ed2164db33060c0303", "data/prereg36/cache/meta.npz": "fa8bf685be64aa100746238caeaa87f917d6adc194d2026d83c7865f61f834d3", "data/prereg36/cache/off.npy": "7c352279359c5035c1693b78a783d534c8b6ac1e7f870e56655b4b764fb2eaf8", "results/prereg36/explore/p36_embed.py": "2f2e732d1309e25e0b70e370583c07b74dd4c692c6490db8fe1b533ecda32c21", "results/prereg36/explore/p36_lib.py": "8df27b37a6b3de786b54af25908d3d91ffe4c030dceb7350d9dd130a14ce241f", "results/prereg36/explore/p36_train.py": "8abc5ef22eba35b70be06c624f7071218d0a4266affe5494d1207f8819271211", "results/prereg36/explore/split_record.json": "2c847257139cd73b9a1808c003ba30b86bcd59eb6860edef8043afc82535e7e3", "results/prereg42/explore/p42_embed.py": "23b60a6f71d1aad5881acd10c87ef2c31a9ed81e6ac2d369c3d29615452b69fd", "results/prereg42/explore/p42_lib.py": "b5045f8b8222e1c3e6891675eb4b3296fe4254b974421c71853ea8a43cea6eea", "results/prereg42/explore/p42_train.py": "ec5d7baf7bf16753cd9d0fb7369e3ff4cb5a9a271d8ed28cf264691bb2359c80", "tools/c10data.py": "2906d97fe41171dc955288d0a6e695fcef905417d18d94cb98f146aac74c6495"}
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()
print(subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout)
tv = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__, torch.cuda.is_available())'], capture_output=True, text=True).stdout.strip()
print('torch', tv)   # in a subprocess: the notebook itself never holds the GPU (p42_train refuses to share it)
assert tv.endswith('True'), 'STOP: 沒有 GPU，請在「變更執行階段類型」選 GPU'
kit = os.path.join(DRIVE, 'C10_p42_kit.zip')
assert os.path.exists(kit), 'STOP: 找不到 ' + kit
ok_flag = os.path.join(WORK, '.kit_ok')
if not (os.path.exists(ok_flag) and open(ok_flag).read() == KIT_SHA):
    assert sha(kit) == KIT_SHA, 'STOP: 套件檔的 SHA-256 不對'
    with zipfile.ZipFile(kit) as z:
        z.extractall(WORK)
    for rel, h in FILES.items():
        assert sha(os.path.join(WORK, rel)) == h, 'STOP: 檔案不對 ' + rel
    open(ok_flag, 'w').write(KIT_SHA)
print('kit ok', len(FILES), 'files')
os.makedirs(os.path.join(DRIVE, VARIANT), exist_ok=True)

In [ ]:
# 1b. prereg51 的程式和 T49 快取：從 C10/p51/overlay_parts/ 接回 overlay zip、核雜湊、解到工作目錄（不用改）
import glob, shutil
P51 = '/content/drive/MyDrive/C10/p51'
OSHA, NPARTS, OFILES = '2729cbb3cf022817a84b217d5cb33a567e63eb5a4c4e55b50e3f69fb2c23f654', 77, {"data/prereg49/cache/cache.json": "b38714742e93787a659932e10cac50ab208813dcfeae1fd82f472c1e45fc9219", "data/prereg49/cache/codes.npy": "2447f0e05a42c3634a3ba16ba2666affeed95e90c4c8724ed29d8a4ee849a09a", "data/prereg49/cache/meta.npz": "c6821af3c8752fb77e1983466ce658d99046669e1bc2b2e4f37ecc9b7bb8ad34", "data/prereg49/cache/off.npy": "f4db9a7caf02fdec324414f7aee6a64ebdc52238257f9c28bda02a468a3553a0", "results/prereg43/explore/p43_run.py": "a767ad139e921de3ff5fe2afe48165c048efa2afdf944bc8402e9ec300f98064", "results/prereg49/explore/p49_lib.py": "3f2b48c0e078cf5d87f6856ab3618b9de31fada8723682868e6ebed2c817c631", "results/prereg49/explore/p49_run.py": "674936efb925fc65beb6f53ac0434473be2b71810d0e408c4cb084dac3617fb2", "results/prereg50/explore/p50_run.py": "12017e38d81097e87bb6f3a778f421ec978e35794af7e1aac8dd9bcc38ef20c9", "results/prereg51/explore/p51_run.py": "7495c9cd31fa3b01ec884235a09c5c5ed38f16a1635b9d10000786d51eb860b7"}
flag = os.path.join(WORK, '.p51_ok')
if not (os.path.exists(flag) and open(flag).read() == OSHA):
    parts = sorted(glob.glob(os.path.join(P51, 'overlay_parts', 'overlay.part*')))
    assert len(parts) == NPARTS, 'STOP: overlay_parts 只有 %d 份' % len(parts)
    tmp = '/content/C10_p51_overlay.zip'
    with open(tmp, 'wb') as o:
        for p in parts:
            with open(p, 'rb') as f:
                o.write(f.read())
    assert sha(tmp) == OSHA, 'STOP: 接回的 overlay zip 雜湊不對'
    with zipfile.ZipFile(tmp) as z:
        z.extractall(WORK)
    for rel, h in OFILES.items():
        assert sha(os.path.join(WORK, rel)) == h, 'STOP: 檔案不對 ' + rel
    open(flag, 'w').write(OSHA)
print('overlay ok', OSHA[:12])

In [ ]:
# 1c. prereg55 的包裝程式 p55_run.py（只把種子改成 1）；核雜湊後寫進工作目錄（不用改）
import base64
H, B64 = 'fa69c69ca09ac83aca52cc5a7def1313928c8b8a731e26e1933fdd5624f83d24', 'IiIicHJlcmVnNTUgKFBMQU4ubWQpOiBjNDkncyByZWNpcGUgd2l0aCBzZWVkIDEgKFYyLCBCMTBDMTI4LCAzMjAsMDAwIHN0ZXBzLCBUNDkgcGxheWVycykgb24gQ29sYWIgRzQgLT4gYW4gZXF1YWxseQpzdHJvbmcgc2Vjb25kIG1lbWJlciBmb3IgcHJlcmVnNTQncyBzY29yZS1sZXZlbCBlbnNlbWJsZS4gT25seSB0aGUgc2VlZCBkaWZmZXJzIGZyb20gYzQ5LgoKICAgIEdQVVBZIHJlc3VsdHMvcHJlcmVnNTUvZXhwbG9yZS9wNTVfcnVuLnB5IHRyYWluIC0tYXJtIFMgWy0tYXV0by1yZXN1bWUgLi4uXSBbLS1vdXQgRElSIC0tcmVjb3JkIEZdCiAgICBHUFVQWSByZXN1bHRzL3ByZXJlZzU1L2V4cGxvcmUvcDU1X3J1bi5weSBlbWJlZCAtLWFybSBTIFtwNDJfZW1iZWQucHkgYXJndW1lbnRzXQoKV3JhcHMgcDUxX3J1bi5weSB1bmNoYW5nZWQ6IGFkZHMgYXJtIFMgKDMyMCwwMDAgc3RlcHMsIEIxMEMxMjgsIGNoZWNrcG9pbnRzIGV2ZXJ5IDUsMDAwIHN0ZXBzKSBhbmQgc2V0cyB0aGUgc2VlZCB0byAxLgoiIiIKaW1wb3J0IG9zCmltcG9ydCBzeXMKCkhFUkUgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkKUkVQTyA9IG9zLnBhdGguYWJzcGF0aChvcy5wYXRoLmpvaW4oSEVSRSwgIi4uIiwgIi4uIiwgIi4uIikpClA1MSA9IG9zLnBhdGguam9pbihSRVBPLCAicmVzdWx0cyIsICJwcmVyZWc1MSIsICJleHBsb3JlIikKaWYgUDUxIG5vdCBpbiBzeXMucGF0aDoKICAgIHN5cy5wYXRoLmluc2VydCgwLCBQNTEpCmltcG9ydCBwNTFfcnVuIGFzIFIgICMgbm9xYTogRTQwMgoKU0VFRCA9IDEKUi5BUk1TWyJTIl0gPSB7InN0ZXBzIjogMzIwMDAwLCAic2l6ZXMiOiAoIkIxMEMxMjgiLCksICJja3B0X2V2ZXJ5IjogNTAwMH0KUi5TRUVEID0gU0VFRAoKCmRlZiBtYWluKGFyZ3Y9Tm9uZSk6CiAgICBhcmd2ID0gbGlzdChzeXMuYXJndlsxOl0gaWYgYXJndiBpcyBOb25lIGVsc2UgYXJndikKICAgIGlmICItLWFybSIgaW4gYXJndiBhbmQgYXJndlthcmd2LmluZGV4KCItLWFybSIpICsgMV0gIT0gIlMiOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoInByZXJlZzU1IHRyYWlucyBhcm0gUyBvbmx5IikKICAgIGlmIGxlbihhcmd2KSA+IDEgYW5kIGFyZ3ZbMF0gPT0gInRyYWluIiBhbmQgIi0tb3V0IiBub3QgaW4gYXJndjoKICAgICAgICBhcmd2ICs9IFsiLS1vdXQiLCBvcy5wYXRoLmpvaW4oUkVQTywgImRhdGEiLCAicHJlcmVnNTUiLCAiUyIpXQogICAgaWYgbGVuKGFyZ3YpID4gMSBhbmQgYXJndlswXSA9PSAidHJhaW4iIGFuZCAiLS1yZWNvcmQiIG5vdCBpbiBhcmd2OgogICAgICAgIGFyZ3YgKz0gWyItLXJlY29yZCIsIG9zLnBhdGguam9pbihIRVJFLCAidHJhaW5fcmVjb3JkX1MuanNvbiIpXQogICAgcmV0dXJuIFIubWFpbihhcmd2KQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBzeXMuZXhpdChtYWluKCkpCg=='
data = base64.b64decode(B64)
assert hashlib.sha256(data).hexdigest() == H, 'STOP: 內嵌的 p55_run.py 不對'
os.makedirs(os.path.join(WORK, 'results/prereg55/explore'), exist_ok=True)
with open(os.path.join(WORK, 'results/prereg55/explore/p55_run.py'), 'wb') as f:
    f.write(data)
P55 = '/content/drive/MyDrive/C10/p55'
out = os.path.join(P55, 'S')
os.makedirs(out, exist_ok=True)
print('p55_run ok', H[:12])

In [ ]:
# 2. 訓練種子 1、320,000 步（約 20 小時；每 5,000 步存到 C10/p55/S/ckpt/；斷線後重按「全部執行」會接續）
workers = str(max(2, min(12, (os.cpu_count() or 4) - 1)))
env = dict(os.environ, PYTHONDONTWRITEBYTECODE='1', PYTHONIOENCODING='utf-8')
def run(args, log):
    with open(log, 'a') as lf:
        p = subprocess.Popen([sys.executable, '-u'] + args, cwd=WORK, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='')
            lf.write(line); lf.flush()
        return p.wait()
rec = os.path.join(out, 'train_record_S.json')
done = os.path.exists(rec) and json.load(open(rec)).get('finished')
if not done:
    rc = run(['results/prereg55/explore/p55_run.py', 'train', '--arm', 'S', '--workers', workers, '--auto-resume', '--out', out, '--record', rec, '--min-free-ram-mb', '2048', '--log-every', '1000'], os.path.join(out, 'train_log.txt'))
    assert rc == 0, 'STOP: 訓練結束碼 %d（4＝記憶體不足或 STOP 檔，重按「全部執行」會接續）' % rc
print('training finished')

In [ ]:
# 3. 打包（不用改）
ck = os.path.join(out, 'ckpt', 'step_320000.pt')
res = os.path.join(P55, 'C10_p55_S_results.zip')
with zipfile.ZipFile(res + '.tmp', 'w', zipfile.ZIP_STORED) as z:
    for name in ('train_record_S.json', 'train_log.txt', 'train_log.jsonl'):
        p = os.path.join(out, name)
        if os.path.exists(p):
            z.write(p, 'S/%s' % name)
    z.write(ck, 'S/ckpt/step_320000.pt')
os.replace(res + '.tmp', res)
drive.flush_and_unmount()
print('完成', os.path.basename(res))